# 05 — Landsat IR Fusion & Decision Tree Baseline

**Learning goal:** Understand multi-sensor fusion. See where SAR and optical agree/disagree.
Establish a no-ML baseline the model must beat.

**Key insight:** SAR alone is ambiguous — Landsat SWIR and TIR remove ambiguity.
Wet soil looks like destroyed buildings in SAR (both are dark). SWIR and TIR
distinguish fire damage from moisture effects because burned areas have elevated
thermal emission and distinct shortwave reflectance signatures.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

try:
    from rasterstats import zonal_stats
except ImportError:
    print("rasterstats not installed — zonal stats cells will be skipped")

try:
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
except ImportError:
    print("sklearn not installed — metrics cells will be skipped")

# Project constants
AOI_BBOX = [-105.16, 39.93, -105.07, 40.01]
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
SAR_DIR = PROCESSED_DIR / "sar"

# Landsat Collection 2 Level 2 scale factors
SR_SCALE = 0.0000275
SR_OFFSET = -0.174
ST_SCALE = 0.00341802
ST_OFFSET = 149.0

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Imports loaded, catalog connected.")

## Load Landsat L2 Bands for Jan 2022 (Post-Fire)

We search Planetary Computer for Landsat Collection 2 Level 2 scenes with <10% cloud cover.
We read four bands windowed to our AOI:
- **Red** (band 4) — for NDVI
- **NIR08** (band 5) — for NDVI and NBR
- **SWIR22** (band 7) — for NBR, and direct fire signal
- **LWIR11** (band 10) — thermal, for TIR anomaly

Scale factors: Surface Reflectance = DN * 0.0000275 - 0.174, Surface Temperature = DN * 0.00341802 + 149.0 (Kelvin).

In [ ]:
def search_landsat(date_range, bbox=AOI_BBOX, max_cloud=10):
    """Search for Landsat C2 L2 scenes."""
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": max_cloud}},
    )
    items = list(search.items())
    print(f"Found {len(items)} Landsat scenes for {date_range}")
    for item in items:
        print(f"  {item.id}  cloud={item.properties.get('eo:cloud_cover', '?')}%")
    return items


def read_landsat_bands(item, bands, bbox=AOI_BBOX):
    """Read specified bands from a Landsat item, windowed to bbox.
    Returns dict of band_name -> 2D array (with scale factors applied).
    """
    result = {}
    transform_out = None

    for band_name in bands:
        href = planetary_computer.sign(item.assets[band_name].href)
        with rasterio.open(href) as src:
            # Transform bbox from EPSG:4326 to the raster's CRS
            bounds_native = transform_bounds("EPSG:4326", src.crs, *bbox)
            window = from_bounds(*bounds_native, transform=src.transform)
            data = src.read(1, window=window).astype(np.float32)

            if transform_out is None:
                transform_out = src.window_transform(window)

            # Apply scale factors
            if band_name == "lwir11":
                data = data * ST_SCALE + ST_OFFSET  # Kelvin
            else:
                data = data * SR_SCALE + SR_OFFSET
                data = np.clip(data, 0, 1)  # Valid reflectance range

            result[band_name] = data
            print(f"  {band_name}: shape={data.shape}, range=[{data.min():.4f}, {data.max():.4f}]")

    return result, transform_out


# --- Post-fire: January 2022 ---
BANDS = ["red", "nir08", "swir22", "lwir11"]

post_items = search_landsat("2022-01-01/2022-01-31")
if not post_items:
    # Widen to Feb if Jan is empty
    post_items = search_landsat("2022-01-01/2022-02-28")

post_item = post_items[0]
print(f"\nUsing post-fire scene: {post_item.id}")
post_bands, post_transform = read_landsat_bands(post_item, BANDS)

# --- Pre-fire: November 2021 ---
pre_items = search_landsat("2021-10-01/2021-11-30")
if not pre_items:
    pre_items = search_landsat("2021-09-01/2021-11-30")

pre_item = pre_items[0]
print(f"\nUsing pre-fire scene: {pre_item.id}")
pre_bands, pre_transform = read_landsat_bands(pre_item, BANDS)

## Compute Spectral Indices

- **NBR** = (NIR - SWIR2) / (NIR + SWIR2) — sensitive to burned areas
- **NDVI** = (NIR - Red) / (NIR + Red) — vegetation health
- **dNBR** = NBR_pre - NBR_post — positive = burn severity
- **TIR anomaly** = TIR_post - TIR_pre — positive = warmer post-fire

In [ ]:
def safe_ratio(a, b):
    """Compute (a-b)/(a+b) safely, returning 0 where denominator is 0."""
    denom = a + b
    result = np.where(denom != 0, (a - b) / denom, 0.0)
    return result.astype(np.float32)


# Post-fire indices
nbr_post = safe_ratio(post_bands["nir08"], post_bands["swir22"])
ndvi_post = safe_ratio(post_bands["nir08"], post_bands["red"])

# Pre-fire indices
nbr_pre = safe_ratio(pre_bands["nir08"], pre_bands["swir22"])
ndvi_pre = safe_ratio(pre_bands["nir08"], pre_bands["red"])

# Change indices
# Positive dNBR = burn severity (vegetation/structure loss)
dnbr = nbr_pre - nbr_post

# TIR anomaly: positive = post-fire surface is warmer
tir_anomaly = post_bands["lwir11"] - pre_bands["lwir11"]

print(f"dNBR     range: [{dnbr.min():.3f}, {dnbr.max():.3f}]")
print(f"NDVI post range: [{ndvi_post.min():.3f}, {ndvi_post.max():.3f}]")
print(f"TIR anom range: [{tir_anomaly.min():.2f}, {tir_anomaly.max():.2f}] K")
print(f"SWIR post range: [{post_bands['swir22'].min():.4f}, {post_bands['swir22'].max():.4f}]")

## Visualize All Four Signals as a 2x2 Grid

Compare SAR VV change (from notebook 04) with the three Landsat-derived signals.
Burn scar should be visible in all four, but with different spatial patterns.

In [ ]:
# Try to load SAR VV change from processed data
sar_vv_change = None
sar_paths = [
    SAR_DIR / "vv_change_db.tif",
    SAR_DIR / "vv_change.tif",
    PROCESSED_DIR / "sar_vv_change.tif",
]
for p in sar_paths:
    if p.exists():
        with rasterio.open(p) as src:
            sar_vv_change = src.read(1)
        print(f"Loaded SAR VV change from {p}")
        break

if sar_vv_change is None:
    print("SAR VV change raster not found on disk.")
    print("Computing proxy from Sentinel-1 via Planetary Computer...")

    sar_search_pre = catalog.search(
        collections=["sentinel-1-grd"],
        bbox=AOI_BBOX,
        datetime="2021-11-01/2021-11-30",
    )
    sar_search_post = catalog.search(
        collections=["sentinel-1-grd"],
        bbox=AOI_BBOX,
        datetime="2022-01-01/2022-01-31",
    )
    sar_pre_items = list(sar_search_pre.items())
    sar_post_items = list(sar_search_post.items())

    if sar_pre_items and sar_post_items:
        def read_sar_vv(item):
            href = planetary_computer.sign(item.assets["vv"].href)
            with rasterio.open(href) as src:
                bounds_native = transform_bounds("EPSG:4326", src.crs, *AOI_BBOX)
                window = from_bounds(*bounds_native, transform=src.transform)
                data = src.read(1, window=window).astype(np.float32)
            return data

        vv_pre = read_sar_vv(sar_pre_items[0])
        vv_post = read_sar_vv(sar_post_items[0])

        # Convert to dB and compute change
        vv_pre_db = 10 * np.log10(np.clip(vv_pre, 1e-10, None))
        vv_post_db = 10 * np.log10(np.clip(vv_post, 1e-10, None))
        sar_vv_change = vv_post_db - vv_pre_db
        print(f"SAR VV change computed: shape={sar_vv_change.shape}")
    else:
        print("Could not find SAR scenes — SAR panel will be blank.")

# --- Plot 2x2 grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel 1: SAR VV change
ax = axes[0, 0]
if sar_vv_change is not None:
    im1 = ax.imshow(sar_vv_change, cmap="RdBu", vmin=-6, vmax=6)
    plt.colorbar(im1, ax=ax, label="dB")
else:
    ax.text(0.5, 0.5, "SAR data not available", ha="center", va="center",
            transform=ax.transAxes)
ax.set_title("SAR VV Change (post - pre dB)")

# Panel 2: dNBR
ax = axes[0, 1]
im2 = ax.imshow(dnbr, cmap="RdYlGn_r", vmin=-0.2, vmax=0.8)
plt.colorbar(im2, ax=ax, label="dNBR")
ax.set_title("dNBR (pre - post)\nPositive = burn severity")

# Panel 3: SWIR2 post-fire
ax = axes[1, 0]
im3 = ax.imshow(post_bands["swir22"], cmap="magma", vmin=0, vmax=0.4)
plt.colorbar(im3, ax=ax, label="Reflectance")
ax.set_title("SWIR2 Post-Fire (Jan 2022)\nBright = exposed soil/ash")

# Panel 4: TIR anomaly
ax = axes[1, 1]
im4 = ax.imshow(tir_anomaly, cmap="hot", vmin=-5, vmax=10)
plt.colorbar(im4, ax=ax, label="K")
ax.set_title("TIR Anomaly (post - pre)\nPositive = warmer surface")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Marshall Fire — Four Damage Signals", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Four-Signal Scatter Matrix Colored by Ground Truth

If we have parcel geometries and FEMA damage labels, compute zonal stats for each
signal per parcel and visualize the scatter matrix. This reveals which signal pairs
best separate destroyed from survived structures.

In [ ]:
# Attempt to load parcels and FEMA damage data
parcel_paths = [
    DATA_DIR / "raw" / "parcels.geojson",
    DATA_DIR / "raw" / "boulder_parcels.geojson",
    DATA_DIR / "tabular" / "parcels.geojson",
]
fema_paths = [
    DATA_DIR / "raw" / "fema_damage.csv",
    DATA_DIR / "tabular" / "fema_damage.csv",
    DATA_DIR / "raw" / "damage_assessment.csv",
]

parcels = None
fema = None

for p in parcel_paths:
    if p.exists():
        parcels = gpd.read_file(p)
        print(f"Loaded parcels from {p}: {len(parcels)} features")
        break

for p in fema_paths:
    if p.exists():
        fema = pd.read_csv(p)
        print(f"Loaded FEMA data from {p}: {len(fema)} rows")
        break

if parcels is not None and fema is not None:
    # Merge parcels with damage labels
    # Try common parcel ID columns
    parcel_id_col = None
    for col in ["parcel_id", "PARCEL_ID", "parcelid", "APN"]:
        if col in parcels.columns and col in fema.columns:
            parcel_id_col = col
            break

    if parcel_id_col is None:
        print("Could not find matching parcel ID column. Columns:")
        print(f"  Parcels: {list(parcels.columns[:10])}")
        print(f"  FEMA: {list(fema.columns[:10])}")
    else:
        merged = parcels.merge(fema, on=parcel_id_col, how="inner")
        print(f"Merged: {len(merged)} parcels with damage labels")

        # Compute zonal stats for each signal
        # We need a temporary rasterio dataset for zonal_stats
        from rasterio.transform import from_bounds as affine_from_bounds

        h, w = dnbr.shape
        aff = affine_from_bounds(*AOI_BBOX, w, h)

        signals = {
            "dnbr_mean": dnbr,
            "swir2_mean": post_bands["swir22"],
            "tir_anom_mean": tir_anomaly,
        }

        merged_4326 = merged.to_crs("EPSG:4326")

        for sig_name, sig_array in signals.items():
            stats = zonal_stats(
                merged_4326, sig_array,
                affine=aff, stats=["mean"], nodata=np.nan,
            )
            merged_4326[sig_name] = [s["mean"] for s in stats]

        if sar_vv_change is not None:
            h_sar, w_sar = sar_vv_change.shape
            aff_sar = affine_from_bounds(*AOI_BBOX, w_sar, h_sar)
            stats_sar = zonal_stats(
                merged_4326, sar_vv_change,
                affine=aff_sar, stats=["mean"], nodata=np.nan,
            )
            merged_4326["vv_change_mean"] = [s["mean"] for s in stats_sar]

        # Determine damage class column
        damage_col = None
        for col in ["damage_class", "damage_category", "DAMAGE", "damage"]:
            if col in merged_4326.columns:
                damage_col = col
                break

        if damage_col:
            signal_cols = [c for c in ["vv_change_mean", "dnbr_mean", "swir2_mean", "tir_anom_mean"]
                          if c in merged_4326.columns]
            plot_df = merged_4326[signal_cols + [damage_col]].dropna()

            fig = pd.plotting.scatter_matrix(
                plot_df[signal_cols],
                c=plot_df[damage_col].astype("category").cat.codes,
                cmap="Set1", alpha=0.5, figsize=(12, 12),
            )
            plt.suptitle("Four-Signal Scatter Matrix (colored by damage class)", fontsize=14)
            plt.tight_layout()
            plt.show()
        else:
            print("No damage class column found in merged data.")
else:
    print("Parcel and/or FEMA damage data not found on disk.")
    print("To run this cell, place parcel geometries and FEMA damage labels in data/raw/.")
    print("Expected files:")
    print("  - data/raw/parcels.geojson (parcel polygons with parcel_id)")
    print("  - data/raw/fema_damage.csv (parcel_id, damage_class)")

## Apply Decision Tree Classifier — No ML, Just Thresholds

This is a hand-tuned threshold classifier based on physical reasoning:

| Rule | Condition | Label |
|------|-----------|-------|
| 1 | SWIR < 0.15 AND dNBR < 0.10 | survived |
| 2 | VV change > -2.0 | survived_fire_exposed |
| 3 | VV change <= -4.0 | total_loss |
| 4 | TIR anomaly >= 1.5 AND SWIR >= 0.25 | total_loss |
| 5 | else | partial_damage |

We apply this to the raster first (pixel-level), then to parcel zonal stats if available.

In [ ]:
def threshold_classifier(vv_change, dnbr_arr, swir2_arr, tir_anom_arr):
    """
    Apply threshold-based damage classification.
    Returns integer class array:
        0 = survived
        1 = survived_fire_exposed
        2 = partial_damage
        3 = total_loss
    """
    # Default: partial_damage
    result = np.full(dnbr_arr.shape, 2, dtype=np.int8)

    # Rule 1: low SWIR and low dNBR = survived (no fire signature)
    survived_mask = (swir2_arr < 0.15) & (dnbr_arr < 0.10)
    result[survived_mask] = 0

    # Rule 2: SAR shows minimal change = survived but fire exposed
    if vv_change is not None:
        # Only apply where not already classified as survived
        fire_exposed = (~survived_mask) & (vv_change > -2.0)
        result[fire_exposed] = 1

    # Rule 3: strong SAR drop = total loss (structure completely gone)
    if vv_change is not None:
        total_sar = vv_change <= -4.0
        result[total_sar] = 3

    # Rule 4: high thermal anomaly + high SWIR = total loss
    total_thermal = (tir_anom_arr >= 1.5) & (swir2_arr >= 0.25)
    result[total_thermal] = 3

    return result


CLASS_NAMES = ["survived", "survived_fire_exposed", "partial_damage", "total_loss"]
CLASS_COLORS = ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c"]

# --- Pixel-level classification ---
# Resample SAR to Landsat grid if shapes differ
if sar_vv_change is not None and sar_vv_change.shape != dnbr.shape:
    from scipy.ndimage import zoom
    zoom_factors = (dnbr.shape[0] / sar_vv_change.shape[0],
                    dnbr.shape[1] / sar_vv_change.shape[1])
    sar_resampled = zoom(sar_vv_change, zoom_factors, order=1)
    print(f"Resampled SAR from {sar_vv_change.shape} to {sar_resampled.shape}")
else:
    sar_resampled = sar_vv_change

classification = threshold_classifier(
    sar_resampled, dnbr, post_bands["swir22"], tir_anomaly
)

# Count pixels per class
print("\nPixel-level classification results:")
for i, name in enumerate(CLASS_NAMES):
    count = np.sum(classification == i)
    pct = 100 * count / classification.size
    print(f"  {name:25s}: {count:7d} pixels ({pct:5.1f}%)")

# Visualize
cmap = mcolors.ListedColormap(CLASS_COLORS)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(classification, cmap=cmap, norm=norm)
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(CLASS_NAMES)
ax.set_title("Threshold-Based Damage Classification (No ML)")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

# --- Parcel-level metrics if ground truth available ---
if parcels is not None and fema is not None and damage_col is not None:
    print("\n--- Parcel-level classification metrics ---")

    # Apply thresholds to parcel zonal stats
    df = merged_4326.copy()
    vv_col = "vv_change_mean" if "vv_change_mean" in df.columns else None

    def classify_parcel(row):
        swir = row.get("swir2_mean", np.nan)
        dnbr_val = row.get("dnbr_mean", np.nan)
        tir = row.get("tir_anom_mean", np.nan)
        vv = row.get("vv_change_mean", np.nan) if vv_col else np.nan

        if pd.isna(swir) or pd.isna(dnbr_val):
            return "partial_damage"
        if swir < 0.15 and dnbr_val < 0.10:
            return "survived"
        if not pd.isna(vv) and vv > -2.0:
            return "survived_fire_exposed"
        if not pd.isna(vv) and vv <= -4.0:
            return "total_loss"
        if not pd.isna(tir) and tir >= 1.5 and swir >= 0.25:
            return "total_loss"
        return "partial_damage"

    df["predicted"] = df.apply(classify_parcel, axis=1)

    # Map ground truth to our class names if needed
    print(f"\nGround truth classes: {df[damage_col].value_counts().to_dict()}")
    print(f"Predicted classes: {df['predicted'].value_counts().to_dict()}")

    # Classification report
    print("\n" + classification_report(df[damage_col], df["predicted"], zero_division=0))

    f1 = f1_score(df[damage_col], df["predicted"], average="weighted", zero_division=0)
    print(f"\nWeighted F1 score: {f1:.3f}")
    print("Expected baseline: ~0.70-0.80. The ML model must beat this.")
else:
    print("\nNo ground truth available for parcel-level metrics.")
    print("The pixel-level map above shows the classifier output.")
    print("Expected F1 when ground truth is available: ~0.70-0.80.")